# Localization Sample : Extended kalman filter (EKF)

---- 

- conda env : [ai_robotics](../../README.md#setup-a-conda-environment)

---

### Ref
- https://github.com/AtsushiSakai/PythonRobotics/
- https://github.com/AtsushiSakai/PythonRobotics/blob/master/Localization/extended_kalman_filter/extended_kalman_filter.py

### Imports and Configuration

In [1]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[0])
# Add this path to sys.path
sys.path.insert(0, parent_dir)


In [2]:
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from utils.plot import plot_covariance_ellipse

# Covariance for EKF simulation
Q = np.diag([0.1, 0.1, np.deg2rad(1.0), 1.0]) ** 2
R = np.diag([1.0, 1.0]) ** 2

# Simulation parameters
INPUT_NOISE = np.diag([1.0, np.deg2rad(30.0)]) ** 2
GPS_NOISE = np.diag([0.5, 0.5]) ** 2
DT = 0.1
SIM_TIME = 50.0


### Motion and Functions

In [3]:
def calc_input():
    v = 1.0
    yawrate = 0.1
    return np.array([[v], [yawrate]])

def motion_model(x, u):
    F = np.array([[1.0, 0, 0, 0],
                  [0, 1.0, 0, 0],
                  [0, 0, 1.0, 0],
                  [0, 0, 0, 0]])
    B = np.array([[DT * math.cos(x[2, 0]), 0],
                  [DT * math.sin(x[2, 0]), 0],
                  [0.0, DT],
                  [1.0, 0.0]])
    return F @ x + B @ u

def observation_model(x):
    H = np.array([[1, 0, 0, 0],
                  [0, 1, 0, 0]])
    return H @ x

def observation(xTrue, xd, u):
    xTrue = motion_model(xTrue, u)
    z = observation_model(xTrue) + GPS_NOISE @ np.random.randn(2, 1)
    ud = u + INPUT_NOISE @ np.random.randn(2, 1)
    xd = motion_model(xd, ud)
    return xTrue, z, xd, ud

def jacob_f(x, u):
    yaw = x[2, 0]
    v = u[0, 0]
    jF = np.array([
        [1.0, 0.0, -DT * v * math.sin(yaw), DT * math.cos(yaw)],
        [0.0, 1.0, DT * v * math.cos(yaw), DT * math.sin(yaw)],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ])
    return jF

def jacob_h():
    return np.array([[1, 0, 0, 0],
                     [0, 1, 0, 0]])

def ekf_estimation(xEst, PEst, z, u):
    # Prediction
    xPred = motion_model(xEst, u)
    jF = jacob_f(xEst, u)
    PPred = jF @ PEst @ jF.T + Q

    # Update
    jH = jacob_h()
    zPred = observation_model(xPred)
    y = z - zPred
    S = jH @ PPred @ jH.T + R
    K = PPred @ jH.T @ np.linalg.inv(S)
    xEst = xPred + K @ y
    PEst = (np.eye(len(xEst)) - K @ jH) @ PPred
    return xEst, PEst


### Animation Setup and Update Function

In [4]:
def run_ekf_animation():
    xEst = np.zeros((4, 1))
    xTrue = np.zeros((4, 1))
    PEst = np.eye(4)
    xDR = np.zeros((4, 1))
    time = 0.0

    hxEst, hxTrue, hxDR, hz = xEst, xTrue, xDR, np.zeros((2, 1))

    # fig, ax = plt.subplots(figsize=(6, 6))
    fig, ax = plt.subplots()
    ax.set_aspect('equal')
    ax.grid(True)

    def init():
        ax.cla()
        ax.grid(True)
        ax.set_aspect('equal')
        return []

    def update(frame):
        nonlocal xEst, xTrue, xDR, PEst, hxEst, hxTrue, hxDR, hz, time
        time += DT
        if time > SIM_TIME:
            ani.event_source.stop()
            return []

        u = calc_input()
        xTrue, z, xDR, ud = observation(xTrue, xDR, u)
        xEst, PEst = ekf_estimation(xEst, PEst, z, ud)

        hxEst = np.hstack((hxEst, xEst))
        hxTrue = np.hstack((hxTrue, xTrue))
        hxDR = np.hstack((hxDR, xDR))
        hz = np.hstack((hz, z))

        # Clean up previous frame
        ax.cla()
        ax.grid(True)
        ax.set_aspect('equal')

        # ---- AUTO SCALE ----
        all_x = np.hstack((hxTrue[0, :], hxEst[0, :], hxDR[0, :]))
        all_y = np.hstack((hxTrue[1, :], hxEst[1, :], hxDR[1, :]))
        x_min, x_max = np.min(all_x), np.max(all_x)
        y_min, y_max = np.min(all_y), np.max(all_y)

        margin_x = (x_max - x_min) * 0.2 + 0.5
        margin_y = (y_max - y_min) * 0.2 + 0.5
        ax.set_xlim(x_min - margin_x, x_max + margin_x)
        ax.set_ylim(y_min - margin_y, y_max + margin_y)

        # ---- DRAW ----
        ax.plot(hz[0, :], hz[1, :], ".g", label='observation')
        ax.plot(hxTrue[0, :], hxTrue[1, :], "-b", label='true')
        ax.plot(hxDR[0, :], hxDR[1, :], "-k", label='dead reckoning')
        ax.plot(hxEst[0, :], hxEst[1, :], "-r", label='estimated')
        plot_covariance_ellipse(xEst[0, 0], xEst[1, 0], PEst, ax=ax)

        ax.legend(loc='lower right', fontsize='small', frameon=True, shadow=True)
        ax.set_title(f"EKF Localization | Time = {time:.1f}s")

        return []

    ani = FuncAnimation(fig, update, frames=int(SIM_TIME / DT),
                        init_func=init, interval=100, blit=False, repeat=False)
    
    plt.close(fig)  # prevent duplicate static plot
    return ani

SIM_TIME = 3 # Update the duration of total simulation time [s]
ani = run_ekf_animation()
HTML(ani.to_html5_video())
